# TML — Free Colab GPU batch runner
Interactive one-shot mode for the free Colab tier: no persistent supervisor, no idle polling. Each run processes the current VPS RapidOCR claim once and exits.


In [ ]:
YEAR=1904
import subprocess
try:
    gpu_line=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],text=True).strip().splitlines()[0]
except Exception as e:
    raise RuntimeError('No GPU detected. In Colab choose Runtime > Change runtime type > GPU.') from e
GPU_NAME,GPU_MEM=[x.strip() for x in gpu_line.rsplit(',',1)]
GPU_MEM_MB=int(float(GPU_MEM))
if GPU_MEM_MB>=70000:
    RAPID_WORKERS,RAPID_DOWNLOADERS=12,8
elif GPU_MEM_MB>=40000:
    RAPID_WORKERS,RAPID_DOWNLOADERS=8,8
elif GPU_MEM_MB>=20000:
    RAPID_WORKERS,RAPID_DOWNLOADERS=6,6
else:
    RAPID_WORKERS,RAPID_DOWNLOADERS=4,4
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
CLAIM=f'{BASE}/00_MANIFEST/colab_active_claims.tsv'
print('FREE_GPU_CONFIG',GPU_NAME,f'{GPU_MEM_MB}MiB','Rapid',RAPID_WORKERS,'downloaders',RAPID_DOWNLOADERS,'YEAR',YEAR,flush=True)


In [ ]:
import os,shutil,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'
RAPID_ENV='/content/tml-rapid-env'
RAPID_PY=f'{RAPID_ENV}/bin/python'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv','paramiko>=3.5,<4'],check=True)
UV=shutil.which('uv'); assert UV
if not os.path.exists(RAPID_PY): subprocess.run([UV,'venv','--seed',RAPID_ENV],check=True)
subprocess.run([RAPID_PY,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'],check=True)
subprocess.run([RAPID_PY,'-c',"import onnxruntime as o; p=o.get_available_providers(); print('RAPID_ENV_READY',o.__version__,p); assert 'CUDAExecutionProvider' in p"],check=True)
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)


In [ ]:
from google.colab import files
import os
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload the dedicated Colab SFTP private key')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
KEY_FILE='/content/tml_colab_key'
open(KEY_FILE,'wb').write(data)
os.chmod(KEY_FILE,0o600)
print('KEY_READY',name,flush=True)


In [ ]:
import subprocess
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip()
print('CODE',commit,flush=True)
cmd=[RAPID_PY,'-u',f'{REPO}/colab/rapid_once_filekey.py','--vps-key-file',KEY_FILE,'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--claim',CLAIM,'--workers',str(RAPID_WORKERS),'--downloaders',str(RAPID_DOWNLOADERS)]
print('STARTING_FREE_RAPID_ONCE',flush=True)
rc=subprocess.run(cmd).returncode
print('FREE_RAPID_ONCE_EXIT',rc,flush=True)
if rc!=0: raise RuntimeError(f'Rapid batch failed rc={rc}')
